In [ ]:
import numpy as np

np.set_printoptions(suppress=True)

params=np.random.uniform(low=-50, high=150, size=20)#generate random parameters

#set the first three values as the maximum, minimum and zero respectively 
#for better view of effect of quantization on these numbers
params[0]=params.max()+1
params[1]=params.min()-1
params[2]=0

params=np.round(params,2)#round to second decimal

print(params)

[146.11 -48.93   0.   140.3   61.96 131.21  -7.4   90.09  13.74  17.19
  54.   -38.63 105.46 131.11 -25.58  42.25  39.55 125.01 -47.93  80.1 ]


In [4]:
#defining the clamp function which helps in keeping the quantization in bounds
def clamp(params_q: np.array, lower_bound: int, upper_bound: int)->np.array:
    params_q[params_q<lower_bound]=lower_bound
    params_q[params_q>upper_bound]=upper_bound
    return params_q

#now the implementation of asymmetric quantization
def asymmetric_quantization(params: np.array, bits: int) -> tuple[np.array,float,int]:
    #find all values of the variables in the quantization formula
    alpha=np.max(params)
    beta=np.min(params)
    scale=(alpha-beta)/(2**bits-1)
    zero=-1*np.round(beta/scale)
    lower_bound=0
    upper_bound=2**bits-1
    #quantize the parameters
    quantized=clamp(np.round(params/scale + zero),lower_bound,upper_bound).astype(np.int32)
    return quantized, scale, zero

def asymmetric_dequantization(params_q: np.array, scale:float, zero:int)-> np.array:
    return(params_q-zero)*scale



In [5]:
#symmetric quantization
def symmetric_quantization(params: np.array, bits: int) -> tuple[np.array,float]:
    alpha=np.max(np.abs(params))
    scale= alpha/(2**(bits-1)-1)
    lower_bound=-2**(bits-1)
    upper_bound=2**(bits-1)-1
    #quantize the parameters
    quantized=clamp(np.round(params/scale),lower_bound,upper_bound).astype(np.int32)
    return quantized, scale

def symmetric_dequantization(params_q: np.array, scale:float)-> np.array:
    return params_q*scale

In [6]:
#this function is for finding the error or loss of precision after dequantization
def quantization_error(params: np.array, params_q: np.array):
    return np.mean((params-params_q)**2)#this is the MSE(mean squared error)


In [7]:
(asymmetric_q, asymmetric_scale,asymmetric_zero)=asymmetric_quantization(params,8)
(symmetric_q, symmetric_scale)=symmetric_quantization(params,8)

print(f"Original: ")
print(np.round(params, 2))
print(" ")

print(f"Asymmetric Scale: {asymmetric_scale}, Zero: {asymmetric_zero}")
print(asymmetric_q)

print(f"Symmetric scale: {symmetric_scale}")
print(symmetric_q)


Original: 
[146.11 -48.93   0.   140.3   61.96 131.21  -7.4   90.09  13.74  17.19
  54.   -38.63 105.46 131.11 -25.58  42.25  39.55 125.01 -47.93  80.1 ]
 
Asymmetric Scale: 0.7648627450980393, Zero: 64.0
[255   0  64 247 145 236  54 182  82  86 135  13 202 235  31 119 116 227
   1 169]
Symmetric scale: 1.150472440944882
[127 -43   0 122  54 114  -6  78  12  15  47 -34  92 114 -22  37  34 109
 -42  70]


In [8]:
#dequantize back to 32 bits
params_deq_asymmetric=asymmetric_dequantization(asymmetric_q,asymmetric_scale, asymmetric_zero)
params_deq_symmetric=symmetric_dequantization(symmetric_q,symmetric_scale)

print(f"Original: ")
print(np.round(params, 2))
print(" ")

print(f"Dequantize Asymmetric: ")
print(np.round(params_deq_asymmetric,2))
print(" ")

print(f"Dequantized Symmetric: ")
print(np.round(params_deq_symmetric,2))

Original: 
[146.11 -48.93   0.   140.3   61.96 131.21  -7.4   90.09  13.74  17.19
  54.   -38.63 105.46 131.11 -25.58  42.25  39.55 125.01 -47.93  80.1 ]
 
Dequantize Asymmetric: 
[146.09 -48.95   0.   139.97  61.95 131.56  -7.65  90.25  13.77  16.83
  54.31 -39.01 105.55 130.79 -25.24  42.07  39.77 124.67 -48.19  80.31]
 
Dequantized Symmetric: 
[146.11 -49.47   0.   140.36  62.13 131.15  -6.9   89.74  13.81  17.26
  54.07 -39.12 105.84 131.15 -25.31  42.57  39.12 125.4  -48.32  80.53]


In [9]:
#calculate the dequantization error
print(f"{"Asymmmetric error: "}{np.round(quantization_error(params, params_deq_asymmetric),2)}")
print(f"{"Symmmetric error: "}{np.round(quantization_error(params, params_deq_symmetric),2)}")

Asymmmetric error: 0.06
Symmmetric error: 0.1
